# 欢迎来到第 2 天实验！


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">开始之前——</h2>
            <span style="color:#f71;">我想花一点时间向你介绍这个课程的实用资源页面。其中包含所有幻灯片的链接。<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            请把这个页面加入书签，我会持续在那里添加更多有用链接。
            </span>
        </td>
    </tr>
</table>

## 首先——让我们谈谈 Chat Completions API

1. 调用 LLM 最简单的方式
2. 之所以叫 Chat Completions，是因为它在说：「这是一段对话，请预测接下来应该是什么」
3. Chat Completions API 由 OpenAI 发明，但太受欢迎了，以至于人人都在用！

### 我们会再次从调用 OpenAI 开始——不过非 OpenAI 用户别担心，你们的时机很快就到！



In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


## 你知道什么是 Endpoint（端点）吗？

如果不知道，请复习 guides 文件夹中的 Technical Foundations 指南

这里还有一个你可能感兴趣的端点……

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

In [ ]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

In [ ]:
response.json()["choices"][0]["message"]["content"]

# 什么是 openai 包？

它被称为 Python 客户端库（Python Client Library）。

它只不过是对向该 http 端点发起完全相同调用的一层封装。

它只是让你能用漂亮的 Python 代码工作，而不必和别扭的 json 对象纠缠。

仅此而已。它是开源且轻量的。有人以为它包含 OpenAI 模型代码——其实没有！



In [ ]:
# 创建客户端

from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content



## 然后发生了很棒的事：

OpenAI 的 Chat Completions API 太受欢迎了，以至于其他模型提供商创建了完全相同的端点。

它们被称为「OpenAI Compatible Endpoints」（OpenAI 兼容端点）。

例如，Google 在这里做了一个：https://generativelanguage.googleapis.com/v1beta/openai/

而 OpenAI 决定慷慨一些：他们说，嘿，你可以直接用我们为 GPT 做的同一个客户端库。我们允许你指定不同的端点 URL 和不同的密钥，来使用其他提供商。

所以你可以这样用：

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

需要明确的是——尽管代码里有 OpenAI，我们只是用这个轻量 Python 客户端库来调用端点——这里并没有涉及 OpenAI 模型。

如果感到困惑，请复习 Guides 文件夹中的 Guide 9！

现在让我们试试看！

## 这是可选的——但如果你想试用 Google Gemini，请访问：

https://aistudio.google.com/

并在以下地址设置你的 API 密钥

https://aistudio.google.com/api-keys

然后把密钥加入 `.env` 文件，修改后务必保存 .env 文件：

`GOOGLE_API_KEY=AIz...`



In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")



In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

## Ollama 也提供 OpenAI 兼容端点

……而且它就在你的本地机器上！

如果下一个单元格没有打印 "Ollama is running"，请打开终端并运行 `ollama serve`

In [ ]:
requests.get("http://localhost:11434").content

### 从 Meta 下载 llama3.2

如果你的电脑配置较低，请改成 llama3.2:1b。

不要用 llama3.3 或 llama4！它们对你的电脑来说太大了……

In [ ]:
!ollama pull llama3.2

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# 【注】Get a fun fact

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

In [ ]:
# 【注】Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

In [ ]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

# 家庭作业练习

把 Day 1 的网页摘要项目升级为使用通过 Ollama 在本地运行的开源模型，而不是 OpenAI

如果你不想使用付费 API，后续所有项目都可以使用这种技术。

**优点：**
1. 无 API 费用——开源
2. 数据不会离开你的电脑

**缺点：**
1. 能力明显弱于前沿模型（Frontier Model）

## Ollama 安装回顾

只需访问 [ollama.com](https://ollama.com) 并安装！

完成后，ollama 服务器应该已经在本地运行。  
如果你访问：  
[http://localhost:11434/](http://localhost:11434/)

你应该看到消息 `Ollama is running`。  

如果没有，打开新的 Terminal（Mac）或 Powershell（Windows）并输入 `ollama serve`  
再在另一个 Terminal（Mac）或 Powershell（Windows）中输入 `ollama pull llama3.2`  
然后再次尝试 [http://localhost:11434/](http://localhost:11434/)。

如果 Ollama 在你的机器上很慢，可以尝试用 `llama3.2:1b` 作为替代。在 Terminal 或 Powershell 中运行 `ollama pull llama3.2:1b`，并把代码中的 `MODEL = "llama3.2"` 改成 `MODEL = "llama3.2:1b"`